# 🏢 Nyaya-Jyoti: AI-Powered Lease Rent Agreement Generator

Welcome to **Nyaya-Jyoti**, your smart legal assistant to generate a personalized **Lease Deed Agreement** in minutes — no legal drafting skills required.

This AI-powered tool uses **GPT-Neo** and **Semantic Similarity Matching** to generate both standard and custom legal clauses. The lease document is dynamically built from your inputs.

---

### 📝 How it works:

1. **Upload the clause CSV registry** and your **lease deed `.docx` template**
2. **Answer simple questions** about the lessor, lessee, rent, term, interest, etc.
3. **Optionally enter a special clause** (e.g., "no subletting allowed") — AI will write it for you
4. **Watch the document get built dynamically**, line by line
5. **Download your finalized Lease Agreement** as a `.docx` file

---

### ⚠️ Instructions:

- Make sure to upload:
  - ✅ Clause prompt CSV file
  - ✅ Word `.docx` template with placeholders like `[Lessor_Name]`, `[Monthly_Rent]`, etc.
- Dates must follow `DD/MM/YYYY` format
- Special clauses will be asked at the end and generated automatically
- Final `.docx` document will download after completion

---

> 🛡️ This tool is part of the *Nyaya-Jyoti* project by Bennett University. Meant for demonstration and educational purposes only.


In [ ]:
# @title
# 🛠️ Install required packages
!pip install -q transformers torch pandas sentence-transformers python-docx

# 📦 Imports
import torch
import pandas as pd
import re
import docx
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util
from google.colab import files
from IPython.display import display, Markdown

# --- Load Clause Prompt Registry ---
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
prompt_df = pd.read_csv(csv_filename)
prompt_df["parameters"] = prompt_df["parameters"].fillna("").astype(str)
prompt_df["parameters"] = prompt_df["parameters"].apply(
    lambda x: ", ".join(sorted(set(p.strip() for p in x.split(",") if p.strip().lower() != "nan")))
)

# --- Load GPT-Neo & Embedding Model ---
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

embedder = SentenceTransformer('all-MiniLM-L6-v2')
instruction_embeddings = embedder.encode(prompt_df['instruction'].tolist(), convert_to_tensor=True)

# --- Clause Generation Utilities ---
def build_prompt(example_1, example_2, instruction):
    return (
        "You are a legal assistant specialized in drafting formal legal clauses.\n"
        f"Example 1:\nClause: {example_1}\nEndClause\n\n"
        f"Example 2:\nClause: {example_2}\nEndClause\n\n"
        f"Now, generate ONLY the legal clause for {instruction}, using formal legal language.\n"
        "Output only the text between the markers 'Clause:' and 'EndClause'.\n\nClause: "
    )

def generate_clause(prompt_text):
    input_ids = tokenizer.encode(prompt_text, return_tensors="pt").to(device)
    output_ids = model.generate(
        input_ids,
        max_length=300,
        temperature=0.35,
        top_k=50,
        top_p=0.85,
        repetition_penalty=1.2,
        do_sample=True,
        num_return_sequences=1,
        pad_token_id=tokenizer.pad_token_id
    )
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    if "EndClause" in generated_text:
        return generated_text.split("Clause:")[1].split("EndClause")[0].strip()
    return generated_text.split("Clause:")[1].strip()

def fill_parameters_dynamic(clause_text, param_string):
    placeholders = set(re.findall(r"{(.*?)}", clause_text))
    defined_params = [p.strip() for p in str(param_string).split(',') if p.strip()]
    combined_params = sorted(placeholders.union(set(defined_params)))
    param_values = {}
    for param in combined_params:
        value = input(f"🧾 Please provide value for '{param}': ").strip()
        param_values[param] = value
    for param, value in param_values.items():
        clause_text = clause_text.replace(f"{{{param}}}", value)
    return clause_text, param_values

def find_best_match_semantic(user_input):
    user_embedding = embedder.encode(user_input, convert_to_tensor=True)
    cosine_scores = util.pytorch_cos_sim(user_embedding, instruction_embeddings)[0]
    best_idx = torch.argmax(cosine_scores).item()
    best_score = cosine_scores[best_idx].item()
    if best_score > 0.4:
        return prompt_df.iloc[best_idx]
    return None

# --- Upload Template ---
print("📂 Upload your Lease Deed DOCX template:")
uploaded_template = files.upload()
template_filename = list(uploaded_template.keys())[0]
doc = docx.Document(template_filename)

# --- Placeholder Descriptions for Lease ---
explanations = {
    "Lessor_Name": "Enter the full name of the Lessor",
    "Lessor_Address": "Enter the complete address of the Lessor",
    "Lessee_Name": "Enter the full name of the Lessee",
    "Lessee_Address": "Enter the complete address of the Lessee",
    "Location": "Enter the city or place where the agreement is signed",
    "Day": "Enter the numeric day of the month",
    "Month": "Enter the month (e.g., April)",
    "Year": "Enter the four-digit year",
    "Lease_Term": "Enter the lease term in years (e.g., 3)",
    "Start_Date": "Enter the lease start date (DD/MM/YYYY)",
    "First_Rent_Date": "Enter the first rent payment date (DD/MM/YYYY)",
    "Monthly_Rent": "Enter the monthly rent amount (e.g., 10000)",
    "Interest_Rate": "Enter the interest rate for delayed payment (e.g., 12)",
    "Arrears_Months": "Enter the number of months allowed for unpaid rent",
    "Notice_Period": "Enter the number of months required for notice",
    "Property_Description": "Describe the leased property in brief",
    "Premises_Location": "Enter the location of the demised premises (e.g., full site address)",
    "Special_Clauses": "Special terms or clauses if any (auto-filled below)",
    "Witness_1_Name": "Enter name of the first witness",
    "Witness_2_Name": "Enter name of the second witness"
}


# --- Fill Placeholders First ---
print("\n📋 Please answer the following questions to populate the lease agreement:")
user_inputs = {}
for placeholder, explanation in explanations.items():
    if placeholder == "Special_Clauses":
        continue
    value = input(f"🖋 {explanation}: ").strip()
    user_inputs[placeholder] = value
    for para in doc.paragraphs:
        if f"[{placeholder}]" in para.text:
            original_text = para.text
            para.text = para.text.replace(f"[{placeholder}]", value)
            display(Markdown(f"**📄 Updated Line:**\n\n`Before:` {original_text}\n\n`After:` {para.text}"))

# --- Special Clause Prompt Comes AFTER Placeholder Filling ---
special_clause_text = ""
print("\n📄 All standard fields are now filled.")
special_query = input("\n💬 Do you have any special clause to include (e.g., 'no commercial activity allowed')?\n📨 Special Request (leave blank if none): ").strip()
if special_query:
    match_row = find_best_match_semantic(special_query)
    if match_row is not None:
        prompt = build_prompt(match_row['example_1'], match_row['example_2'], match_row['instruction'])
        raw_clause = generate_clause(prompt)
        special_clause_text, _ = fill_parameters_dynamic(raw_clause, match_row.get('parameters', ''))
    else:
        print("⚠️ Could not process your request. Proceeding without it.")

# --- Insert Special Clause in Template ---
if special_clause_text:
    for para in doc.paragraphs:
        if "[Special_Clauses]" in para.text:
            para.text = para.text.replace("[Special_Clauses]", special_clause_text)
            display(Markdown(f"**📄 Inserted Special Clause:** {para.text}"))
            break

# --- Save Final Lease Deed ---
output_file = "completed_lease_deed.docx"
doc.save(output_file)
files.download(output_file)
print(f"\n✅ Lease Deed saved as: {output_file}")


Saving Lease_Agreement_Clause_Registry.csv to Lease_Agreement_Clause_Registry (1).csv
📂 Upload your Lease Deed DOCX template:


Saving lease_deed_template_final.docx to lease_deed_template_final (1).docx

📋 Please answer the following questions to populate the lease agreement:
🖋 Enter the full name of the Lessor: Amit


**📄 Updated Line:**

`Before:` This Deed of Lease is made at [Location] this [Day] day of [Month], [Year], between [Lessor_Name], resident of [Lessor_Address], hereinafter called 'The Lessor' of the One Part and [Lessee_Name], resident of [Lessee_Address], hereinafter called 'The Lessee' of the Other Part.

`After:` This Deed of Lease is made at [Location] this [Day] day of [Month], [Year], between Amit, resident of [Lessor_Address], hereinafter called 'The Lessor' of the One Part and [Lessee_Name], resident of [Lessee_Address], hereinafter called 'The Lessee' of the Other Part.

**📄 Updated Line:**

`Before:` Signed by Shri [Lessor_Name], within named Lessor.

`After:` Signed by Shri Amit, within named Lessor.

🖋 Enter the complete address of the Lessor: D2 1303 Abc road


...

🖋 Enter name of the first witness: sanskar
🖋 Enter name of the second witness: Bilal

📄 All standard fields are now filled.

💬 Do you have any special clause to include (e.g., 'no commercial activity allowed')?
📨 Special Request (leave blank if none): No commerical activity allowed
🧾 Please provide value for 'Allowed_Use_Type': only for living purpose


**📄 Inserted Special Clause:** Commercial use of premises is strictly prohibited and leads to termination.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Lease Deed saved as: completed_lease_deed.docx
